In [1]:
# imports
import re
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity


In [2]:
documents = [
    "Machine learning allows computers to learn patterns from data.",
    "Deep learning uses neural networks with multiple hidden layers.",
    "Natural language processing helps machines understand human language.",
    "Computer vision enables AI systems to interpret images and videos.",
    "Supervised learning requires labelled training data.",
    "Unsupervised learning finds hidden structures in unlabelled data.",
    "Reinforcement learning trains agents using rewards and penalties.",
    "Decision trees are simple yet powerful machine learning algorithms.",
    "Random forests combine multiple decision trees for better accuracy.",
    "Support vector machines are effective for classification problems.",
    "TF-IDF measures the importance of words in a document.",
    "Cosine similarity calculates similarity between text vectors.",
    "Embeddings capture semantic meaning beyond keyword matching.",
    "Large language models generate human-like text responses.",
    "Tokenization splits text into smaller units called tokens.",
    "Stopword removal eliminates common but unimportant words.",
    "Stemming reduces words to their root forms.",
    "Lemmatization converts words into meaningful base forms.",
    "Overfitting happens when a model memorizes training data.",
    "Cross-validation helps evaluate machine learning models reliably."
]

In [3]:
# NLP PIPELINE CLASS
class NLPPipeline:
    
    def __init__(self):
        self.vectorizer = TfidfVectorizer(stop_words='english')
    
    def preprocess(self, text):
        text = text.lower()
        text = re.sub(r'[^a-zA-Z\s]', '', text)
        return text
    
    def fit_transform(self, documents):
        cleaned_docs = [self.preprocess(doc) for doc in documents]
        return self.vectorizer.fit_transform(cleaned_docs)
    
    def transform(self, text):
        cleaned_text = self.preprocess(text)
        return self.vectorizer.transform([cleaned_text])

In [4]:
pipeline = NLPPipeline()

# Create TF-IDF matrix
corpus_matrix = pipeline.fit_transform(documents)

print("TF-IDF Matrix Shape:", corpus_matrix.shape)


TF-IDF Matrix Shape: (20, 105)


In [5]:
# RETRIEVAL FUNCTION
def retrieve(query, corpus_matrix, top_k=3, threshold=0.1):
    
    # Vectorize query
    query_vector = pipeline.transform(query)
    
    # Compute cosine similarity
    similarities = cosine_similarity(query_vector, corpus_matrix).flatten()
    
    # Get top matches
    top_indices = similarities.argsort()[::-1][:top_k]
    
    # Relevance threshold check
    if similarities[top_indices[0]] < threshold:
        return "No relevant document found."
    
    results = []
    
    for idx in top_indices:
        results.append({
            "document": documents[idx],
            "score": round(similarities[idx], 3)
        })
    
    return results

In [6]:
# TEST QUERIES
queries = [
    "What is deep learning?",
    "How do machines understand language?",
    "Explain neural networks",
    "How are images processed in AI?",
    "What is tokenization in NLP?",
    
    # Ambiguous Queries
    "training models",
    "text understanding",
    
    # Out-of-domain Queries
    "Best football players",
    "How to cook pasta",
    
    # Synonym / Failure Query
    "word representations"
]

for i, query in enumerate(queries, 1):
    
    print("\n" + "="*60)
    print(f"Query {i}: {query}")
    print("="*60)
    
    results = retrieve(query, corpus_matrix)
    
    if isinstance(results, str):
        print(results)
    else:
        for rank, result in enumerate(results, 1):
            print(f"\nRank {rank}")
            print("Document:", result["document"])
            print("Similarity Score:", result["score"])


Query 1: What is deep learning?

Rank 1
Document: Deep learning uses neural networks with multiple hidden layers.
Similarity Score: 0.442

Rank 2
Document: Supervised learning requires labelled training data.
Similarity Score: 0.138

Rank 3
Document: Machine learning allows computers to learn patterns from data.
Similarity Score: 0.126

Query 2: How do machines understand language?

Rank 1
Document: Natural language processing helps machines understand human language.
Similarity Score: 0.708

Rank 2
Document: Support vector machines are effective for classification problems.
Similarity Score: 0.202

Rank 3
Document: Large language models generate human-like text responses.
Similarity Score: 0.195

Query 3: Explain neural networks

Rank 1
Document: Deep learning uses neural networks with multiple hidden layers.
Similarity Score: 0.539

Rank 2
Document: Cross-validation helps evaluate machine learning models reliably.
Similarity Score: 0.0

Rank 3
Document: Overfitting happens when a mo